In [1]:
import pandas as pd
import os
from typing import Iterable
import re


In [ ]:
class SurveyProcessor:
	def __init__(self, raw_dir: str, processed_dir: str, situations_map: dict[str, list[int]]):
		self.raw_dir = raw_dir
		self.processed_dir = processed_dir
		self.situations_dir = os.path.join(
			self.processed_dir,
			"situations",
		)
		self.situations_map = situations_map

		os.makedirs(self.situations_dir, exist_ok=True)
	
	@staticmethod
	def clean_text(text):
		if pd.isna(text):
			return text

		out = str(text)

		out = re.sub(r"\(.*?\)", "", out)
		out = re.sub(r"\*.*?\*", "", out)
		out = re.sub(r"[\"“”]", "", out)
		out = re.sub(r"\s+", " ", out)

		return out.strip()    

	@staticmethod
	def detect_blocks(columns: Iterable[str]):
		# Cada situación aborda 1-2 columnas, pero todas terminan con un campo de texto libre, con la oración (1 frase).
		free_text_cols = [
			c for c in columns
			if "(1 frase)" in c
		]

		blocks = []
		current_block = []

		# Se identifican las columnas que pertenecen a cada situación en base a lo anterior.
		for col in columns:
			current_block.append(col)

			if col in free_text_cols:
				blocks.append(current_block)
				current_block = []

		return blocks
	
	def _export_situations(self, clean_df: pd.DataFrame, situations: list[int]):
		for situation in situations:
			cols = [
				col for col in clean_df.columns
				if col.startswith(f"situation_{situation}_")
			]

			subdf = clean_df[
				[
					"id",
					"timestamp",
					"age",
					"gender",
					"sexual_orientation",
					*cols,
				]
			].copy()

			subdf = subdf.rename(columns={
				f"situation_{situation}_choice": "choice",
				f"situation_{situation}_text": "text",
			})

			path = os.path.join(
				self.situations_dir,
				f"situation_{situation}.csv",
			)

			subdf.to_csv(path, encoding="utf-8", index=False)
	
	def process_file(self, filename: str) -> pd.DataFrame:
		raw_path = os.path.join(self.raw_dir, filename)

		print(f"Procesando: {raw_path}")

		situations = self.situations_map[filename]

		# Se lee el cuestionario
		df = pd.read_csv(raw_path, encoding="utf-8")

		# Se juntan las columnas de orientación sexual
		df["Orientación sexual"] = (
			df["Orientación Sexual"]
			.fillna(df["Orientación Sexual.1"])
		)

		meta_cols = {
			"Marca temporal": "timestamp",
			"Edad": "age",
			"Género": "gender",
			"Orientación sexual": "sexual_orientation",
		}


		drop_cols = [
			"Marca temporal",
			"Edad",
			"Género",
			"Orientación Sexual",
			"Orientación Sexual.1",
			"Orientación sexual",
		]

		# Se crea una dataframe con los metadatos
		meta = (
			df[list(meta_cols.keys())]
			.rename(columns=meta_cols)
			.copy()
		)

		# Se crea un dataframe con las situaciones
		data = df.drop(columns=drop_cols)

		# Se detectan los diferentes bloques de las situaciones
		blocks = self.detect_blocks(data.columns)

		# Como las situaciones están duplicadas en función del género del acosador, se parten a la mitad
		n = len(blocks) // 2

		first_half = blocks[:n]
		second_half = blocks[n:]

		# Se asigna a cada número de situación sus dos blqoues correspondientes 
		merged = {
			situation: [first, second]
			for situation, first, second in zip(
				situations,
				first_half,
				second_half,
			)
		}

		rows = []
		# Se itera por el dataframe de las situaciones
		for i, survey_row in df.iterrows():
			# Se agregan los metadatos de cada usuario
			row = {
				col: meta.loc[i, col]
				for col in meta.columns
			}

			# Se recorren las diferentes situaciones
			for idx, pairs in merged.items():
				# Se recorre cada bloque de cada situación
				for columns in pairs:
					# Una situación puede esta formada por: elección y campo libre o solo campo libre
					labels = ["text"] if len(columns) <= 1 else ["choice", "text"]

					# Se crean filas con cada usuario, con las diferentes preguntas de la situación
					for column, label in zip(columns, labels):
						value = survey_row[column]

						if pd.notna(value):
							row[f"situation_{idx}_{label}"] = value

			rows.append(row)

		# Se crea el nuevo df, donde se evita la duplicación de situaciones por el género del acosador
		clean_df = pd.DataFrame(rows)

		clean_df.insert(0, "id", range(len(clean_df)))

		# Se obtienen las columnas de campo libre y se limpia el texto de forma básica
		text_columns = [
			col for col in clean_df.columns
			if "text" in col
		]

		clean_df[text_columns] = (
			clean_df[text_columns]
			.apply(lambda col: col.map(self.clean_text))
		)

		processed_path = os.path.join(
			self.processed_dir,
			filename,
		)

		# Se guarda el cuestionario procesado
		clean_df.to_csv(processed_path, encoding="utf-8", index=False)

		# Se divide el cuestionario en las diferentes situaciones y se guardan
		self._export_situations(clean_df, situations)

		print(f"Guardado: {processed_path}")

		return clean_df

	def process_all(self, extension: str = ".csv"):
		files = [
			f for f in os.listdir(self.raw_dir)
			if f.endswith(extension)
		]

		print(f"Archivos encontrados: {len(files)}")

		results = {}

		for filename in files:
			try:
				clean_df = self.process_file(filename)
				results[filename] = clean_df

			except Exception as e:
				print(f"Error procesando {filename}: {e}")

		return results
	

In [ ]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")
processed_dir = os.path.join(data_dir, "split")


In [4]:
processor = SurveyProcessor(
	raw_dir=raw_dir,
	processed_dir=processed_dir,
	situations_map = {
		"survey_1.csv": [1, 4, 7, 10],
		"survey_2.csv": [2, 5, 8, 11],
		"survey_3.csv": [3, 6, 9, 12],
	}
)


In [5]:
results = processor.process_all()
print(len(results))


Archivos encontrados: 3
Procesando: ./data\raw\survey_1.csv
Guardado: ./data\structured\survey_1.csv
Procesando: ./data\raw\survey_2.csv
Guardado: ./data\structured\survey_2.csv
Procesando: ./data\raw\survey_3.csv
Guardado: ./data\structured\survey_3.csv
3


In [6]:
df = processor.process_file("survey_1.csv")
df.head()


Procesando: ./data\raw\survey_1.csv
Guardado: ./data\structured\survey_1.csv


,id,timestamp,age,gender,sexual_orientation,situation_1_choice,situation_1_text,situation_4_choice,situation_4_text,situation_7_choice,situation_7_text,situation_10_text
0,0,18/04/2026 21:20:28,21,Femenino,Heterosexual,Amigable y cordial.,"Hola, soy Patri! Encantada de conocerte","Preocupado, mostrando interés por cómo se encu...",¿Enserio? ¿Pero te has tomado algo para el dol...,"Bien, divirtiéndote y disfrutando del momento.",Super bien! La verdad es que todos son bastant...,Heyy sigues despierto? Estás haciendo algo ahora?
1,1,19/04/2026 15:15:37,24,Femenino,Homosexual,Amigable y cordial.,"Holaaa buenas, soy Nombre, acabo de llegar y e...","Indiferente, sin mostrar especial interés.","Ayyy qué rollo, bueno al menos ya estás bien","Más o menos, no del todo cómodo.",no aguanto más me quiero ir xd,"volví de la fiesta, sigues por ahí?"
2,2,19/04/2026 15:44:41,23,Otros,Homosexual,Tímido y reservado.,"Hola, Laura, encantado.","Preocupado, mostrando interés por cómo se encu...","Ay, estas bien?","Más o menos, no del todo cómodo.",No muy bien. No conozco a casi nadie y no sé q...,"Holi, ya he vuelto de la fiesta, qué tal te ha..."
3,3,19/04/2026 20:57:37,20,Otros,Homosexual,Tímido y reservado.,Emmm. Hola.,"Preocupado, mostrando interés por cómo se encu...",oooo Todo guay seguro?,"Bien, divirtiéndote y disfrutando del momento.",Guay la verdad q la gente es muy maja aqui,"Estoy en casa ya, que tal??"
4,4,20/04/2026 15:15:28,23,Femenino,Homosexual,Amigable y cordial.,Hola Laura yo soy Rocío encantada,"Preocupado, mostrando interés por cómo se encu...",Ay no pobre cómo estás ahora ?,"Bien, divirtiéndote y disfrutando del momento.",Muy bien!,Hola Lucía! Tú día qué tal ha ido ?
